# Exercise 4.3 — SFF Plateau Value for the GUE

**Chapter 4: Scrambling Dynamics** &nbsp;|&nbsp; *Section 4.3: Spectral Form Factor*

---

## Motivation

The **spectral form factor** (SFF) $K(t) = |Z(it)|^2 = |\mathrm{Tr}(e^{-iHt})|^2$ is the spectral diagnostic of quantum chaos.  It probes pair correlations of energy levels.  The SFF exhibits three regimes: a **slope** (dip below $D^2$), a **ramp** (linear in $t$, reflecting level repulsion), and a **plateau** at $K = D$.  This exercise derives the plateau value from time-averaging and relates it to the dephasing of off-diagonal terms.

## Preliminaries / Toolbox

Before diving into the solution, recall the following concepts:

**1. Spectral Form Factor (SFF):** Provides a proxy for the analytic continuation of the partition function to study late-time chaos geometry. $K(t) = |Z(i t)|^2 = |\mathrm{Tr}(e^{-i H t})|^2 = \sum_{m,n} e^{-i(E_m - E_n)t}$.

**2. Level Repulsion:** In chaotic models (like GUE), eigenvalues repel each other ($P(\Delta E) \sim \Delta E^\beta \to 0$). They never perfectly cross.

**3. Time Averaging:** Off-diagonal terms $e^{-i(E_m - E_n)t}$ strongly oscillate and vanish under time average, isolating diagonal contributions.

**4. Coherence vs Dephasing:** $K(0) = D^2$ shows full coherence, while $K \to D$ indicates full dephasing into mutually orthogonal eigenvectors.



## Exercise Statement

For a $D \times D$ GUE Hamiltonian with eigenvalues $\{E_n\}$: $K(t) = \sum_{m,n} e^{-it(E_m - E_n)}$.

**(a)** Show the long-time average $\overline{K} = D$.

**(b)** Explain why $K_{\mathrm{plateau}} = D$ (not $D^2$) and interpret the ratio $K(0)/\overline{K}$.

## Solution

### Part (a): Time-averaging the SFF

Time-average each term separately:

$$
\overline{K} = \lim_{T\to\infty}\frac{1}{T}\int_0^T K(t)\,dt = \sum_{m,n} \lim_{T\to\infty} \frac{1}{T}\int_0^T e^{-i(E_m - E_n)t}\,dt.
$$

**Diagonal terms** ($m = n$): The integrand is 1.  Each contributes 1.  There are $D$ such terms.

**Off-diagonal terms** ($m \neq n$): Let $\omega_{mn} = E_m - E_n \neq 0$.  Then:

$$
\frac{1}{T}\int_0^T e^{-i\omega_{mn} t}\,dt = \frac{1 - e^{-i\omega_{mn} T}}{i\omega_{mn} T} \to 0 \quad \text{as } T \to \infty.
$$

GUE level repulsion ensures $\omega_{mn} \neq 0$ with probability 1 (eigenvalues don't coincide).

$$
\boxed{\overline{K} = D.}
$$

### Part (b): Interpretation

At $t = 0$: all $D^2$ terms contribute coherently — $K(0) = D^2$.

At late times: the $D(D-1)$ off-diagonal terms dephase (oscillate with incommensurate frequencies and cancel upon averaging), leaving only the $D$ diagonal terms.

The ratio:

$$
\frac{K(0)}{\overline{K}} = \frac{D^2}{D} = D
$$

quantifies the **coherent enhancement** at $t = 0$.  This dephasing from $D^2$ to $D$ is the spectral signature of quantum chaos: the transition from coherent to incoherent behavior as the system explores its energy landscape.

---
## Symbolic Verification (SymPy)

In [ ]:
import sympy as sp

D, t = sp.symbols('D t', positive=True)

# SFF: K(t) = |Tr(e^{-iHt})|^2 = sum_{m,n} e^{-i(E_m - E_n)t}
# Time average: <K> = sum_{m,n} delta(E_m, E_n) * 1
# For non-degenerate spectrum: <K> = D (only diagonal terms survive)
print('Spectral Form Factor time average:')
print(f'  K(0) = |Tr(I)|^2 = D^2')
print(f'  <K>_T = D  (diagonal terms only, non-degenerate)')
print(f'  Ratio K(0)/<K> = D')

# Numerical check
for n in range(2, 8):
    d = 2**n
    print(f'  N={n}: K(0)={d**2}, <K>={d}, ratio={d}')

---
## Numerical Verification

In [ ]:
import numpy as np

np.random.seed(42)

for D in [4, 8, 16, 32]:
    A = np.random.randn(D,D) + 1j*np.random.randn(D,D)
    H = (A + A.conj().T) / (2*np.sqrt(D))
    E = np.linalg.eigvalsh(H)
    
    # SFF at 1000 time points and average
    times = np.linspace(10, 500, 1000)
    K_vals = [abs(np.sum(np.exp(-1j*E*t)))**2 for t in times]
    K_bar = np.mean(K_vals)
    
    print(f"D={D:3d}: K(0)={D**2:5d},  K̄={K_bar:8.2f},  K̄/D={K_bar/D:.3f}")
    assert abs(K_bar/D - 1) < 0.15

print("\nSFF plateau at K̄ = D confirmed. ✓")

---
## Visual Pedagogical Proof

In [ ]:
import matplotlib.pyplot as plt

np.random.seed(45)
D = 32
A = np.random.randn(D,D) + 1j*np.random.randn(D,D)
H = (A + A.conj().T) / (2*np.sqrt(D))
E = np.linalg.eigvalsh(H)

times_log = np.logspace(-1, 3, 300)
K = np.array([abs(np.sum(np.exp(-1j*E*t)))**2 for t in times_log])

window = 10
K_smooth = np.convolve(K, np.ones(window)/window, mode='valid')
times_smooth = times_log[window//2 : -window//2 + 1]

plt.rcParams['figure.dpi'] = 150
plt.figure(figsize=(8, 4.5))
plt.loglog(times_log, K, color='gray', alpha=0.4, label='SFF $K(t)$')
plt.loglog(times_smooth, K_smooth, color='royalblue', linewidth=2, label='Smoothed $K(t)$')
plt.axhline(D, color='crimson', linestyle='--', linewidth=2, label=r'Plateau $ar{K} = D$')
plt.axhline(D**2, color='black', linestyle=':', label=r'Initial $K(0) = D^2$')

plt.xlabel(r"Time $t$")
plt.ylabel(r"Spectral Form Factor $K(t)$")
plt.title("SFF for GUE: Slope, Ramp, and Plateau")
plt.legend()
plt.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()